## Imports

In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

## Load Data

In [44]:
import pandas as pd

df = pd.read_csv("../raw_data/hotel_bookings.csv")

## Custom Feature Engineering Transformer

In [ ]:
def transform(self, X):
    X = X.copy()
    # company -> binary
    if "company" in X.columns:
        X["company_booking"] = X["company"].notna().astype(int)
    # agent -> binary
    if "agent" in X.columns:
        X["has_agent"] = X["agent"].notna().astype(int)
    # parking -> binary
    if "required_car_parking_spaces" in X.columns:
        X["has_parking"] = (X["required_car_parking_spaces"] > 0).astype(int)
    # family flags
    if "children" in X.columns:
        X["has_children"] = (X["children"].fillna(0) > 0).astype(int)
    if "babies" in X.columns:
           X["has_babies"] = (X["babies"].fillna(0) > 0).astype(int)

       # total stay
      if {"stays_in_weekend_nights", "stays_in_week_nights"}.issubset(X.columns):
          X["total_stay"] = (
                X["stays_in_weekend_nights"] + X["stays_in_week_nights"]
          )

        # convert reservation_status_date to datetime only if you want to inspect it,
        # but we will drop it later because it leaks future information

        return X

In [ ]:
def additional_features(df):

    df = df.copy()

    df["total_stay"] = (
        df["stays_in_weekend_nights"] +
        df["stays_in_week_nights"]
    )

    df["total_guests"] = (
        df["adults"] +
        df["children"].fillna(0) +
        df["babies"]
    )

    df["is_family"] = (
        (df["children"].fillna(0) > 0) |
        (df["babies"] > 0)
    ).astype(int)

    df["weekend_ratio"] = (
        df["stays_in_weekend_nights"] /
        (df["total_stay"] + 1)
    )

    df["company_booking"] = df["company"].notna().astype(int)
    df["has_agent"] = df["agent"].notna().astype(int)

    df["arrival_month"] = pd.to_datetime(
        df["arrival_date_month"],
        format="%B"
    ).dt.month

    df["high_adr"] = (
        df["adr"] > df["adr"].median()
    ).astype(int)

    return df

## Log Transform Selected Skewed Numeric Columns

In [46]:
def log_transform_selected(X):
    X = X.copy()

    # X comes in as numpy array if used too late in pipeline,
    # so keep this inside a dataframe-aware step before ColumnTransformer
    skewed_cols = ["lead_time", "adr", "days_in_waiting_list", "total_stay"]

    for col in skewed_cols:
        if col in X.columns:
            X[col] = np.log1p(X[col])

    return X

## Columns by type

In [ ]:
target_col = "is_canceled"

drop_cols = [
    "reservation_status",       # leakage -> drop
    "reservation_status_date",  # leakage ? /
    "company",                  # replaced by company_booking
    "agent"                     # replaced by has_agent
]

numeric_features = [
    "lead_time",
    "arrival_date_year",
    "arrival_date_week_number",
    "arrival_date_day_of_month",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "booking_changes",
    "days_in_waiting_list",
    "adr",
    "total_of_special_requests",
    "total_stay"
]

binary_features = [
    "is_repeated_guest",
    "company_booking",
    "has_agent",
    "has_parking",
    "has_children",
    "has_babies"
]

categorical_features = [
    "hotel",
    "arrival_date_month",
    "meal",
    "country",
    "market_segment",
    "distribution_channel",
    "reserved_room_type",
    "assigned_room_type",
    "deposit_type",
    "customer_type"
]

## Pipelines

In [48]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

binary_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent"))
    # no scaling
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

## Full Preprocessor

In [ ]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("bin", binary_pipeline, binary_features),
    ("cat", categorical_pipeline, categorical_features)
])

## Full Modeling Pipeline

In [50]:
from sklearn.linear_model import LogisticRegression

model_pipeline = Pipeline([
    ("feature_engineering", HotelFeatureEngineer()),
    ("log_transform", FunctionTransformer(log_transform_selected)),
    ("drop_leakage", FunctionTransformer(
        lambda X: X.drop(columns=[col for col in drop_cols if col in X.columns])
    )),
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])